In [4]:
import glob
from pathlib import Path

import pandas as pd

# Ambil semua file CSV di folder data, kecuali dataset Valentine
all_files = sorted(glob.glob("data/*.csv"))
files = [f for f in all_files if "valentine" not in Path(f).name.lower()]

if not files:
    raise FileNotFoundError("Tidak ada file CSV yang valid di folder data/ setelah filter.")

print("File yang diproses:")
for f in files:
    print("-", Path(f).name)

df_list = []

for file in files:
    # Coba beberapa kombinasi agar tahan terhadap perbedaan delimiter/encoding
    attempts = [
        {"sep": ",", "encoding": "utf-8"},
        {"sep": ";", "encoding": "utf-8"},
        {"sep": ",", "encoding": "latin1"},
        {"sep": ";", "encoding": "latin1"},
    ]

    last_error = None
    for opt in attempts:
        try:
            df = pd.read_csv(file, sep=opt["sep"], encoding=opt["encoding"])
            df_list.append(df)
            print(f"OK: {Path(file).name} -> {len(df)} baris")
            break
        except Exception as e:
            last_error = e
    else:
        raise ValueError(f"Gagal membaca file: {file}\nError terakhir: {last_error}")

# Gabungkan semua dataframe
merged_df = pd.concat(df_list, ignore_index=True, sort=False)

# Keep hanya kolom yang diminta
selected_columns = [
    "id_str",
    "full_text",
    "created_at",
    "lang",
    "favorite_count",
    "retweet_count",
    "reply_count",
    "quote_count",
]

missing_columns = [col for col in selected_columns if col not in merged_df.columns]
if missing_columns:
    raise KeyError(f"Kolom tidak ditemukan: {missing_columns}")

final_df = merged_df[selected_columns]

# Simpan hasil akhir versi asli
output_path = "Scraping_X.csv"
final_df.to_csv(output_path, index=False, encoding="utf-8")

# Normalisasi mention @username menjadi @USER pada full_text
normalized_df = final_df.copy()
normalized_df["full_text"] = (
    normalized_df["full_text"]
    .fillna("")
    .astype(str)
    .str.replace(r"@[A-Za-z0-9_]{1,15}", "@USER", regex=True)
)

normalized_output_path = "Scraping_X_normalized.csv"
normalized_df.to_csv(normalized_output_path, index=False, encoding="utf-8")

print(f"Selesai! Total file: {len(files)}")
print(f"Total baris gabungan: {len(final_df)}")
print(f"Kolom akhir: {list(final_df.columns)}")
print(f"File output asli: {output_path}")
print(f"File output normalisasi: {normalized_output_path}")

File yang diproses:
- Bahlil.csv
- NU.csv
- jawa (2).csv
- merge.csv
- pemerintah (1).csv
- ramadhan (1).csv
- uang (1).csv
OK: Bahlil.csv -> 500 baris
OK: NU.csv -> 500 baris
OK: jawa (2).csv -> 500 baris
OK: merge.csv -> 2602 baris
OK: pemerintah (1).csv -> 500 baris
OK: ramadhan (1).csv -> 500 baris
OK: uang (1).csv -> 500 baris
Selesai! Total file: 7
Total baris gabungan: 5602
Kolom akhir: ['id_str', 'full_text', 'created_at', 'lang', 'favorite_count', 'retweet_count', 'reply_count', 'quote_count']
File output asli: Scraping_X.csv
File output normalisasi: Scraping_X_normalized.csv
